# Unified Perception — OWL-SAM Adapter Training

Trains the `OwlToSamAdapter` that bridges OWL-ViT-B encoder output `(B, 576, 768)` to SAM-compatible features `(B, 256, 64, 64)`.

**Phase B:** Architecture validation with synthetic features (real OWL/SAM encoders in Phase C).

**GPU:** T4 (free Colab) is sufficient. Enable it via `Runtime → Change runtime type → T4 GPU`.

## 1. Check GPU

In [1]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Clone Repo

In [2]:
!git clone https://github.com/sutharsan-311/unified-perception.git
%cd unified-perception
!ls

fatal: destination path 'unified-perception' already exists and is not an empty directory.
/content/unified-perception
ADAPTER_DESIGN_SUMMARY.md  PHASE_B_TRAINING.md
benchmark_adapter.py	   README.md
checkpoints		   RESEARCH_UNIFIED_ENCODER.md
data			   test_adapter_quick.py
logs			   tests
nanosam			   train.py
PHASE_A_COMPLETE.md	   unified_perception_colab.ipynb


## 3. Install Dependencies

In [3]:
!pip install -q torchvision Pillow

## 4. Download COCO val2017 (~780 MB)

We use `val2017` (5000 images) for both train and validation since it's much smaller than `train2017` (18 GB). Phase B uses synthetic features anyway, so the actual image content doesn't affect training quality.

In [4]:
import os
os.makedirs('data/coco', exist_ok=True)

# Download val2017 (~780 MB)
!wget -q --show-progress -O data/coco/val2017.zip http://images.cocodataset.org/zips/val2017.zip
!unzip -q data/coco/val2017.zip -d data/coco/
!rm data/coco/val2017.zip

# Use val2017 as both splits
!ln -s val2017 data/coco/train2017 2>/dev/null || true

print(f"Images available: {len(list(__import__('pathlib').Path('data/coco/val2017').glob('*.jpg')))}")

data/coco/val2017.z 100%[===================>] 777.80M  53.4MB/s    in 15s     
replace data/coco/val2017/000000212226.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: Images available: 5000


## 5. Verify Adapter Shape

In [5]:
import sys
sys.path.insert(0, '.')

from nanosam.adapter import OwlToSamAdapter

adapter = OwlToSamAdapter()
print(f"Parameters:      {adapter.get_parameter_count():,}")
print(f"Memory (FP32):   {adapter.get_parameter_count() * 4 / 1e6:.2f} MB")

# Shape check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.randn(2, 576, 768).to(device)
adapter = adapter.to(device)
out = adapter(x)
print(f"\nInput:  {x.shape}")
print(f"Output: {out.shape}  (expected [2, 256, 64, 64])")
assert out.shape == (2, 256, 64, 64)
print("✓ Shape OK")

Parameters:      525,568
Memory (FP32):   2.10 MB

Input:  torch.Size([2, 576, 768])
Output: torch.Size([2, 256, 64, 64])  (expected [2, 256, 64, 64])
✓ Shape OK


## 6. Train

Uses `COCO_SMALL` preset tuned for Colab T4:
- 1000 images, 5 epochs, batch size 4, FP16
- ~15–20 min on T4

In [ ]:
!python train.py \
  --dataset-dir data/coco \
  --num-images 1000 \
  --validation-size 100 \
  --batch-size 4 \
  --num-epochs 5 \
  --learning-rate 1e-3 \
  --fp16 \
  --device cuda \
  --checkpoint-dir checkpoints \
  --num-workers 2 \
  --log-interval 10


OWL-SAM Adapter Training

Configuration:
  Dataset: COCO train2017
  Images: 1000
  Batch size: 4
  Epochs: 5
  Learning rate: 0.001
  Device: cuda
  Loss: 0.5*MSE + 0.5*Cosine

Creating adapter...
  Parameters: 525,568
  Memory (FP32): 2.00MB

Creating data loaders...
Loaded 1000 images from train2017
Loaded 100 images from val2017

Creating trainer...

Starting training...


Epoch 1/5
Epoch 0 [0/250] Loss: 1.498694 (Avg: 1.498694) LR: 1.02e-04
Epoch 0 [10/250] Loss: 1.498749 (Avg: 1.498988) LR: 1.20e-04
Epoch 0 [20/250] Loss: 1.495126 (Avg: 1.497877) LR: 1.38e-04
Epoch 0 [30/250] Loss: 1.494632 (Avg: 1.496977) LR: 1.56e-04
Epoch 0 [40/250] Loss: 1.494469 (Avg: 1.496197) LR: 1.74e-04
Epoch 0 [50/250] Loss: 1.489757 (Avg: 1.495191) LR: 1.92e-04
Epoch 0 [60/250] Loss: 1.487975 (Avg: 1.494080) LR: 2.10e-04
Epoch 0 [70/250] Loss: 1.483931 (Avg: 1.492759) LR: 2.28e-04
Epoch 0 [80/250] Loss: 1.479031 (Avg: 1.491296) LR: 2.46e-04
Epoch 0 [90/250] Loss: 1.472638 (Avg: 1.489584) LR: 2.64e-04


## 7. Check Checkpoints

In [ ]:
!ls -lh checkpoints/

## 8. Save to Google Drive (optional)

Mount Drive to persist checkpoints across sessions.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r checkpoints/ /content/drive/MyDrive/unified-perception-checkpoints/
# print('Saved to Google Drive!')

## 9. Load & Verify Best Checkpoint

In [ ]:
import glob

checkpoints = sorted(glob.glob('checkpoints/*.pth'))
print('Saved checkpoints:')
for ck in checkpoints:
    print(f'  {ck}')

# Load final checkpoint (always saved at end of training)
final_ckpt = 'checkpoints/adapter_phase_b_final.pth'
if __import__('os').path.exists(final_ckpt):
    ck = torch.load(final_ckpt, map_location=device, weights_only=False)
    print(f'\nFinal checkpoint:')
    print(f'  Epoch:        {ck["epoch"]}')
    print(f'  Train losses: {[round(l, 4) for l in ck["train_losses"]]}')
    print(f'  Val losses:   {[round(l, 4) for l in ck["val_losses"]]}')
    if ck["val_losses"]:
        print(f'  Best val loss: {min(ck["val_losses"]):.6f}')
else:
    print('No checkpoint found.')